# HERO Time Series Clustering (Step 3)

This interactive notebook allows you to explore and visualize the time series clustering results for regional and national food insecurity trajectories.

The analysis implements three main similarity measures without interpolation:
1. **DTW (Dynamic Time Warping)**: Shape-based similarity for aligned country-level series.
2. **pycatch22**: Feature-based similarity capturing dynamics and complexity.
3. **NCD (Normalized Compression Distance)**: Compression-based parameter-free dissimilarity.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure local modules can be loaded
sys.path.append(os.getcwd())
import config
import similarity_utils
import run_clustering

## 1. Load and Prepare the Dataset
We load the complete, imputed parquet dataset: `merged_adm1_wide_norm_f_imputed.parquet`.

In [ ]:
df_meta, regions_ts, regions_multivariate_ts = run_clustering.load_and_prepare_data()
df_meta.head()

## 2. Country-Level Clustering (Admin 1)
Select an eligible country (e.g. `'SOM'`, `'AFG'`, `'KEN'`, or `'SDN'`) and perform shape-based (DTW) and feature-based clustering.

In [ ]:
country_code = 'SOM' # Change to 'AFG', 'SDN', etc.
c_meta = df_meta[df_meta["country"] == country_code].reset_index(drop=True)
c_series = {row["adm1_pcode"]: regions_ts[row["key"]] for _, row in c_meta.iterrows()}

print(f"Processing country: {country_code} with {len(c_meta)} regions")

# 1. Compute DTW and NCD matrices
dtw_matrix = similarity_utils.compute_distance_matrix(c_series, method="dtw", w=12)
ncd_matrix = similarity_utils.compute_distance_matrix(c_series, method="ncd")

# Plot DTW distance Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(dtw_matrix, annot=True, fmt=".2f", cmap="viridis", xticklabels=c_meta["region_name"], yticklabels=c_meta["region_name"])
plt.title(f"{country_code} Pairwise DTW Distance Heatmap")
plt.tight_layout()
plt.show()

### Plot Cluster Medoids
We identify the medoide of each cluster to see the actual representative profiles of food insecurity dynamics.

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score

condensed_dtw = squareform(dtw_matrix, checks=False)
Z_dtw = linkage(condensed_dtw, method='average')

# Evaluate Silhouette score for best k
for k in [2, 3, 4]:
    if k < len(c_series):
        lbls = fcluster(Z_dtw, t=k, criterion='maxclust')
        score = silhouette_score(dtw_matrix, lbls, metric="precomputed")
        print(f"DTW Hierarchical clustering with k={k}: Silhouette = {score:.4f}")

# Choose k=2
labels_k2 = fcluster(Z_dtw, t=2, criterion='maxclust')
labels_dict = dict(zip(c_meta["adm1_pcode"], labels_k2))

plt.figure(figsize=(10, 5))
for cid in np.unique(labels_k2):
    medoid = similarity_utils.find_cluster_medoid(c_series, labels_dict, cid, dtw_matrix)
    if medoid:
        s = c_series[medoid]
        plt.plot(range(len(s)), s.values, label=f"Cluster {cid} Medoid ({medoid})", marker='o', linewidth=2.5)

plt.title(f"{country_code} Cluster Medoids (DTW shape-based, k=2)")
plt.xlabel("Observations (Chronological index)")
plt.ylabel("IPC Phase 3+ %")
plt.legend()
plt.show()

## 3. Global Univariate and Multivariate Clustering
Cluster all 475 regions globally using concatenated pycatch22 features extracted from multivariate drivers.

In [ ]:
print("Global univariate and multivariate results have been computed.")
print(f"Outputs and plots are available at: {config.OUTPUT_DIR}")